In [ ]:
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds

from sklearn.utils import shuffle
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

from tensorflow.keras.applications import imagenet_utils
from tensorflow.keras.utils import load_img, img_to_array

np.random.seed(42)
tf.random.set_seed(42)

script_dir = (
    Path(__file__).resolve().parent
    if "__file__" in globals()
    else Path.cwd()
)

cache_dir = script_dir / "Cache"
dataset_cache = cache_dir / "Datasets"
model_cache = cache_dir / "Models"

dataset_cache.mkdir(parents=True, exist_ok=True)
model_cache.mkdir(parents=True, exist_ok=True)

print("Cache Directory:", cache_dir)
physical_devices = tf.config.list_physical_devices("GPU")
for device in physical_devices:
    try:
        tf.config.experimental.set_memory_growth(
            device,
            True
        )
    except RuntimeError as error:
        print(error)

def plot_confusion_matrix(
    cm,
    classes,
    normalize=False,
    title="Confusion Matrix",
    cmap=plt.cm.Blues
):

    if normalize:
        row_sums = cm.sum(
            axis=1,
            keepdims=True
        )
        cm = np.divide(
            cm.astype(float),
            row_sums,
            out=np.zeros_like(cm, dtype=float),
            where=row_sums != 0
        )
    plt.figure(figsize=(10, 8))

    plt.imshow(
        cm,
        interpolation="nearest",
        cmap=cmap
    )
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))

    plt.xticks(
        tick_marks,
        classes,
        rotation=45
    )
    plt.yticks(
        tick_marks,
        classes
    )
    threshold = cm.max() / 2 if cm.size else 0

    for i, j in itertools.product(
        range(cm.shape[0]),
        range(cm.shape[1])
    ):
        value = (
            f"{cm[i, j]:.2f}"
            if normalize
            else str(cm[i, j])
        )
        plt.text(
            j,
            i,
            value,
            horizontalalignment="center",
            color="white" if cm[i, j] > threshold else "black"
        )

    plt.tight_layout()
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.show()

rng = np.random.default_rng(42)

train_samples = []
train_labels = []

test_samples = []
test_labels = []

for i in range(50):

    train_samples.append(rng.integers(13, 65))
    train_labels.append(1)

    train_samples.append(rng.integers(65, 101))
    train_labels.append(0)

for i in range(1000):

    train_samples.append(rng.integers(13, 65))
    train_labels.append(0)

    train_samples.append(rng.integers(65, 101))
    train_labels.append(1)

for i in range(10):

    test_samples.append(rng.integers(13, 65))
    test_labels.append(1)

    test_samples.append(rng.integers(65, 101))
    test_labels.append(0)

for i in range(20):

    test_samples.append(rng.integers(13, 65))
    test_labels.append(0)

    test_samples.append(rng.integers(65, 101))
    test_labels.append(1)

train_samples = np.array(train_samples)
train_labels = np.array(train_labels)

test_samples = np.array(test_samples)
test_labels = np.array(test_labels)

train_samples, train_labels = shuffle(
    train_samples,
    train_labels,
    random_state=42
)

test_samples, test_labels = shuffle(
    test_samples,
    test_labels,
    random_state=42
)


scaler = MinMaxScaler(
    feature_range=(0, 2)
)

scaled_train_samples = scaler.fit_transform(
    train_samples.reshape(-1, 1)
)
scaled_test_samples = scaler.transform(
    test_samples.reshape(-1, 1)
)

def create_age_model():

    model = tf.keras.Sequential([
        tf.keras.layers.Input(
            shape=(1,)
        ),
        tf.keras.layers.Dense(
            16,
            activation="relu"
        ),
        tf.keras.layers.Dense(
            32,
            activation="relu"
        ),
        tf.keras.layers.Dense(
            2,
            activation="softmax"
        )
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.01
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

age_model_path = model_cache / "Keras Model.keras"

if age_model_path.exists():
    age_model = tf.keras.models.load_model(
        age_model_path
    )
else:
    age_model = create_age_model()
    age_model.fit(
        scaled_train_samples,
        train_labels,
        validation_split=0.1,
        batch_size=10,
        epochs=45,
        shuffle=True,
        verbose=2
    )

    age_model.save(age_model_path)

age_model.summary()

age_predictions = age_model.predict(
    scaled_test_samples,
    verbose=0
)
age_predictions = np.argmax(
    age_predictions,
    axis=1
)
age_cm = confusion_matrix(
    test_labels,
    age_predictions
)
plot_confusion_matrix(
    age_cm,
    ["No Side Effects", "Side Effects"],
    title="Age Classification Confusion Matrix"
)


age_weights_path = model_cache / "Keras Model.weights.h5"
age_model.save_weights(
    age_weights_path
)

age_json = age_model.to_json()
loaded_json_model = tf.keras.models.model_from_json(
    age_json
)

print("\nAge Model Weights:")
print(age_model.get_weights())

image_size = (224, 224)
batch_size = 10

food_cache = dataset_cache / "Food101"

food_cache.mkdir(
    parents=True,
    exist_ok=True
)

(train_data, official_test_data), food_info = tfds.load(
    "food101",
    split=["train", "validation"],
    as_supervised=True,
    with_info=True,
    data_dir=str(food_cache)
)

burger_id = food_info.features["label"].str2int(
    "hamburger"
)

fries_id = food_info.features["label"].str2int(
    "french_fries"
)

def filter_food_classes(image, label):

    return tf.logical_or(
        tf.equal(label, burger_id),
        tf.equal(label, fries_id)
    )

train_data = train_data.filter(
    filter_food_classes
)
official_test_data = official_test_data.filter(
    filter_food_classes
)

def preprocess_food(image, label):
    image = tf.image.resize(
        image,
        image_size
    )
    image = tf.cast(
        image,
        tf.float32
    )
    label = tf.cast(
        tf.equal(label, fries_id),
        tf.int32
    )
    return image, label

train_data = train_data.map(
    preprocess_food,
    num_parallel_calls=tf.data.AUTOTUNE
)
official_test_data = official_test_data.map(
    preprocess_food,
    num_parallel_calls=tf.data.AUTOTUNE
)

train_ds = train_data.take(1200)
valid_ds = train_data.skip(1200).take(300)
test_ds = official_test_data


train_ds = train_ds.cache(
    str(dataset_cache / "food101_train")
)
valid_ds = valid_ds.cache(
    str(dataset_cache / "food101_valid")
)
test_ds = test_ds.cache(
    str(dataset_cache / "food101_test")
)

train_ds = train_ds.shuffle(
    1200,
    reshuffle_each_iteration=True
)
train_ds = train_ds.batch(
    batch_size
).prefetch(
    tf.data.AUTOTUNE
)
valid_ds = valid_ds.batch(
    batch_size
).prefetch(
    tf.data.AUTOTUNE
)
test_ds = test_ds.batch(
    batch_size
).prefetch(
    tf.data.AUTOTUNE
)
food_classes = [
    "Burger",
    "Fries"
]

def plot_images(images, labels=None):

    fig, axes = plt.subplots(
        1,
        len(images),
        figsize=(20, 4)
    )
    axes = np.atleast_1d(axes).flatten()
    for i, (img, ax) in enumerate(zip(images, axes)):
        ax.imshow(
            np.clip(img.numpy() / 255.0, 0, 1)
        )
        if labels is not None:
            ax.set_title(
                food_classes[int(labels[i].numpy())]
            )
        ax.axis("off")

    plt.tight_layout()
    plt.show()

food_images, food_labels = next(iter(train_ds))

plot_images(
    food_images,
    food_labels
)

def create_food_cnn():

    model = tf.keras.Sequential([
        tf.keras.layers.Input(
            shape=(224, 224, 3)
        ),
        tf.keras.layers.Rescaling(
            1.0 / 255
        ),
        tf.keras.layers.Conv2D(
            32,
            (3, 3),
            activation="relu",
            padding="same"
        ),
        tf.keras.layers.MaxPooling2D(
            (2, 2)
        ),
        tf.keras.layers.Conv2D(
            64,
            (3, 3),
            activation="relu",
            padding="same"
        ),
        tf.keras.layers.MaxPooling2D(
            (2, 2)
        ),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(
            2,
            activation="softmax"
        )
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

cnn_path = model_cache / "CNN.keras"

if cnn_path.exists():
    cnn = tf.keras.models.load_model(
        cnn_path
    )
else:
    cnn = create_food_cnn()
    cnn.fit(
        train_ds,
        validation_data=valid_ds,
        epochs=20,
        verbose=2
    )
    cnn.save(cnn_path)

cnn.summary()
cnn_loss, cnn_accuracy = cnn.evaluate(
    test_ds,
    verbose=0
)
print("\nFood CNN Test Accuracy :", cnn_accuracy)
cnn_predictions = cnn.predict(
    test_ds,
    verbose=0
)
cnn_predictions = np.argmax(
    cnn_predictions,
    axis=1
)
food_true_labels = np.concatenate([
    labels.numpy()
    for images, labels in test_ds
])
food_cm = confusion_matrix(
    food_true_labels,
    cnn_predictions
)
plot_confusion_matrix(
    food_cm,
    food_classes,
    title="Food CNN Confusion Matrix"
)

vgg_path = model_cache / "VGG16 Burger Fries.keras"

def preprocess_vgg(image, label):
    image = tf.keras.applications.vgg16.preprocess_input(
        image
    )
    return image, label

vgg_train_ds = train_ds.map(
    preprocess_vgg,
    num_parallel_calls=tf.data.AUTOTUNE
)
vgg_valid_ds = valid_ds.map(
    preprocess_vgg,
    num_parallel_calls=tf.data.AUTOTUNE
)
vgg_test_ds = test_ds.map(
    preprocess_vgg,
    num_parallel_calls=tf.data.AUTOTUNE
)

if vgg_path.exists():
    vgg_model = tf.keras.models.load_model(
        vgg_path
    )
else:
    vgg_base = tf.keras.applications.VGG16(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3)
    )
    vgg_base.trainable = False
    vgg_model = tf.keras.Sequential([
        tf.keras.layers.Input(
            shape=(224, 224, 3)
        ),
        vgg_base,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(
            2,
            activation="softmax"
        )
    ])
    vgg_model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    vgg_model.fit(
        vgg_train_ds,
        validation_data=vgg_valid_ds,
        epochs=10,
        verbose=2
    )
    vgg_model.save(vgg_path)

vgg_model.summary()
vgg_loss, vgg_accuracy = vgg_model.evaluate(
    vgg_test_ds,
    verbose=0
)
print("\nVGG16 Test Accuracy :", vgg_accuracy)
vgg_predictions = vgg_model.predict(
    vgg_test_ds,
    verbose=0
)
vgg_predictions = np.argmax(
    vgg_predictions,
    axis=1
)
vgg_cm = confusion_matrix(
    food_true_labels,
    vgg_predictions
)
plot_confusion_matrix(
    vgg_cm,
    food_classes,
    title="VGG16 Confusion Matrix"
)

def count_params(model):
    trainable_params = np.sum([
        np.prod(weight.shape)
        for weight in model.trainable_weights
    ])
    non_trainable_params = np.sum([
        np.prod(weight.shape)
        for weight in model.non_trainable_weights
    ])
    return {
        "trainable_params": int(trainable_params),
        "non_trainable_params": int(non_trainable_params),
        "total_params": int(
            trainable_params + non_trainable_params
        )
    }

print("\nCNN Parameters :")
print(count_params(cnn))
print("\nVGG16 Parameters :")
print(count_params(vgg_model))

mobilenet_path = model_cache / "MobileNet.keras"
if mobilenet_path.exists():
    mobilenet_model = tf.keras.models.load_model(
        mobilenet_path
    )
else:
    mobilenet_model = tf.keras.applications.MobileNet(
        weights="imagenet"
    )
    mobilenet_model.save(mobilenet_path)

mobilenet_path = model_cache / "MobileNet.keras"
if mobilenet_path.exists():
    mobilenet_model = tf.keras.models.load_model(
        mobilenet_path
    )
else:
    mobilenet_model = tf.keras.applications.MobileNet(
        weights="imagenet"
    )

    mobilenet_model.save(mobilenet_path)

sample_urls = {
    "Cat": (
        "https://storage.googleapis.com/"
        "download.tensorflow.org/example_images/"
        "grace_hopper.jpg"
    ),
    "Flower": (
        "https://storage.googleapis.com/"
        "download.tensorflow.org/example_images/"
        "YellowLabradorLooking_new.jpg"
    ),
    "Dog": (
        "https://storage.googleapis.com/"
        "download.tensorflow.org/example_images/"
        "Pixar_Inside_Out.jpg"
    )
}

sample_cache = dataset_cache / "MobileNetSamples"
sample_cache.mkdir(
    parents=True,
    exist_ok=True
)
sample_images = {}
for name, url in sample_urls.items():
    image_path = tf.keras.utils.get_file(
        fname=f"{name}.jpg",
        origin=url,
        cache_dir=str(dataset_cache),
        cache_subdir="MobileNetSamples"
    )
    sample_images[name] = image_path

def prepare_mobilenet_image(image_path):

    image = load_img(
        image_path,
        target_size=(224, 224)
    )
    image_array = img_to_array(image)
    image_array = np.expand_dims(
        image_array,
        axis=0
    )
    return tf.keras.applications.mobilenet.preprocess_input(
        image_array
    )

def predict_mobilenet(name, image_path):

    image = prepare_mobilenet_image(
        image_path
    )
    predictions = mobilenet_model.predict(
        image,
        verbose=0
    )
    results = imagenet_utils.decode_predictions(
        predictions,
        top=3
    )
    print(f"\nPredictions for {name} :")
    for _, label, confidence in results[0]:
        print(
            f"{label} : {confidence:.2%}"
        )

for name, image_path in sample_images.items():
    image = load_img(
        image_path,
        target_size=(224, 224)
    )
    plt.imshow(image)
    plt.title(name)
    plt.axis("off")
    plt.show()
    predict_mobilenet(
        name,
        image_path
    )

sample_images = []

for images, labels in train_ds.take(1):
    sample_images = images[:4].numpy()

augmentations = [
    ("Original", lambda img: img),
    ("Resized", lambda img: tf.image.resize(img, (128, 128))),
    ("Horizontal Flip", lambda img: tf.image.flip_left_right(img)),
    ("Rotation", lambda img: tf.image.rot90(img)),
    ("Brightness", lambda img: tf.image.adjust_brightness(img, 0.2)),
    ("Contrast", lambda img: tf.image.adjust_contrast(img, 1.8)),
    ("Grayscale", lambda img: tf.image.grayscale_to_rgb(
        tf.image.rgb_to_grayscale(img)
    )),
    ("Zoom", lambda img: tf.image.central_crop(img, 0.7))
]

fig, axes = plt.subplots(
    len(sample_images),
    len(augmentations),
    figsize=(20, 12)
)

fig.suptitle("Food 101 Data Augmentation", fontsize=18)
for row, original_image in enumerate(sample_images):
    for col, (augmentation_name, augmentation_function) in enumerate(augmentations):
        augmented_image = augmentation_function(
            tf.convert_to_tensor(original_image)
        )
        augmented_image = tf.clip_by_value(
            augmented_image, 0, 255
        )
        augmented_image = tf.cast(
            augmented_image, tf.uint8
        )
        axes[row, col].imshow(augmented_image.numpy())
        axes[row, col].set_title(augmentation_name)
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()